Missing company tags from ChatGPT

But first we merge the tags to chatgpt_glassdoor_reviews_with_sector.csv

In [5]:
import pandas as pd
import os

print(os.getcwd())

reviews = pd.read_csv("../../data/raw/glassdoor-companies-reviews.csv")
lookup = pd.read_csv("../../data/company_tagger/chatgpt_company_tagged.csv")

df = reviews.merge(
    lookup,
    on="company_name",
    how="left"
)

df["sector"].isna().sum()

df.to_csv(
    "../../data/company_tagger/chatgpt_glassdoor_reviews_with_sector.csv",
    index=False
)

c:\Users\roddu\glassdoor-company-reviews\src\python


In [6]:
missing = df[df["sector"].isna()][["company_name", "sector"]]

print(missing)

                    company_name sector
45              Metro Systems VA    NaN
48                           BAT    NaN
51               The Menta Group    NaN
52    Hallmark Aviation Services    NaN
61                           UPS    NaN
...                          ...    ...
996               Signant Health    NaN
997                        Yotpo    NaN
998   Avery Products Corporation    NaN
999      Axiom Real Time Metrics    NaN
1000                       Honda    NaN

[682 rows x 2 columns]


In [7]:
missing_companies = (
    df[df["sector"].isna()][["company_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(missing_companies)

                   company_name
0              Metro Systems VA
1                           BAT
2               The Menta Group
3    Hallmark Aviation Services
4                           UPS
..                          ...
611                       Brady
612              Signant Health
613                       Yotpo
614  Avery Products Corporation
615     Axiom Real Time Metrics

[616 rows x 1 columns]


First ChatGPT attempt was so bad with 616 empty mappings. 
Round 2. Saving the 693 companies to a txt with just the names, then feeding it in batches of 100 names.

In [27]:
companies = (
    reviews["company_name"]
    .dropna()
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

print(len(companies))


# companies.to_csv(
#     "../../data/company_tagger/company_names.txt",
#     index=False,
#     header=False
# )

693


In [29]:
lookup2 = pd.read_csv("../../data/company_tagger/chatgpt_company_sector_mapping.csv")


df2 = reviews.merge(
    lookup2,
    on="company_name",
    how="left"
)


df2.to_csv(
    "../../data/company_tagger/chatgpt_glassdoor_reviews_with_sector_revised.csv",
    index=False
)

In [30]:
missing2 = df2[df2["sector"].isna()][["company_name", "sector"]]

print(missing2)

        company_name sector
964  Dot Foods, Inc.    NaN


Only 1 mapping missing, edited to map to Manufacturing

In [36]:
reviews2 = pd.read_csv("../../data/raw/glassdoor-companies-reviews.csv")
lookup3 = pd.read_csv("../../data/company_tagger/chatgpt_company_sector_mapping.csv")

df3 = reviews2.merge(
    lookup3,
    on="company_name",
    how="left"
)
missing3 = df3[df3["sector"].isna()][["company_name", "sector"]]

print(missing3)

Empty DataFrame
Columns: [company_name, sector]
Index: []


Mapping complete

In [44]:
summ = pd.read_csv("../../data/processed/chatgpt_glassdoor_reviews_with_sector_revised.csv")

sector_summary = (
    summ["sector"]
    .value_counts(dropna=False)
    .rename_axis("sector")
    .reset_index(name="count")
)

sector_summary["percent"] = (
    sector_summary["count"] / sector_summary["count"].sum() * 100
).round(2)

print(f"Row count: {len(summ)}")
print(sector_summary)

Row count: 1000
                   sector  count  percent
0                 Finance    169     16.9
1              Technology    160     16.0
2              Healthcare    122     12.2
3           Manufacturing    117     11.7
4   Professional Services     98      9.8
5                  Retail     77      7.7
6             Hospitality     37      3.7
7               Marketing     35      3.5
8          Transportation     31      3.1
9               Education     31      3.1
10          Energy/Mining     27      2.7
11     Telecommunications     23      2.3
12           Construction     16      1.6
13                  Media     16      1.6
14              Nonprofit     13      1.3
15             Government     10      1.0
16            Real Estate     10      1.0
17                  Legal      6      0.6
18            Agriculture      1      0.1
19                    NaN      1      0.1


In [46]:
missing_company = summ[
    summ["company_name"].isna() | 
    (summ["company_name"].str.strip() == "")
]

print(missing_company.index)

RangeIndex(start=0, stop=0, step=1)
